# 🎭 Free Talking Photo & Lip-Sync Video Studio
### ১০০% ফ্রি গুগল কোলাব GPU ব্যবহার করে আপনার ছবি ও কবিতা থেকে সম্পূর্ণ নিখুঁত ভিডিও তৈরি করুন

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaalikolom-ux/video-project/blob/main/colab_talking_photo_lipsync.ipynb)

---
### 📌 ব্যবহারের নিয়ম (মাত্র ১টি ক্লিকেই সব হবে):
**নিচের সেলের বামপাশে থাকা Play (▶️) বাটনে একবার ক্লিক করুন (অথবা কীবোর্ডে `Ctrl + Enter` চাপুন)।**

স্বয়ংক্রিয়ভাবে সব সেটআপ হবে, ছবি ও অডিও লোড হবে, লিপ-সিঙ্ক ভিডিও রেন্ডার হবে এবং ভিডিওটি আপনার কম্পিউটারে ডাউনলোড হয়ে যাবে!

In [ ]:
# ====================================================================
# 🎬 ONE-CLICK COMPLETE TALKING PHOTO & LIP-SYNC GENERATOR
# ====================================================================
import os
import sys
import glob
import urllib.request
from IPython.display import HTML, display
from base64 import b64encode
from google.colab import files

print("====================================================================")
print("🚀 ধাপ ১/৩: পরিবেশ ও এআই মডেল সেটআপ হচ্ছে... (১-২ মিনিট সময় লাগবে)")
print("====================================================================")

# 1. GPU Check
!nvidia-smi

# 2. Clone SadTalker repository if not exists
if not os.path.exists("/content/SadTalker"):
    !git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker

%cd /content/SadTalker

# 3. Install compatible NumPy 1.x & dependencies
!pip install -q "numpy<2.0.0" "numba<0.60"
!pip install -q -r requirements.txt
!pip install -q facexlib gfpgan yacs

# 4. Apply compatibility patches
!find /content/SadTalker -name "*.py" -exec sed -i 's/np.VisibleDeprecationWarning/getattr(np, "VisibleDeprecationWarning", UserWarning)/g' {} + 2>/dev/null || true
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' $(python -c "import site; print(site.getsitepackages()[0])")/basicsr/data/degradations.py 2>/dev/null || true

# 5. Download model weights
!mkdir -p checkpoints gfpgan/weights
!wget -nc -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar -O ./checkpoints/mapping_00109-model.pth.tar
!wget -nc -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00229-model.pth.tar -O ./checkpoints/mapping_00229-model.pth.tar
!wget -nc -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors -O ./checkpoints/SadTalker_V0.0.2_256.safetensors
!wget -nc -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors -O ./checkpoints/SadTalker_V0.0.2_512.safetensors
!wget -nc -q https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth -O ./gfpgan/weights/alignment_WFLW_4HG.pth
!wget -nc -q https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth -O ./gfpgan/weights/detection_Resnet50_Final.pth
!wget -nc -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth -O ./gfpgan/weights/GFPGANv1.4.pth
!wget -nc -q https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth -O ./gfpgan/weights/parsing_parsenet.pth

print("\n====================================================================")
print("📸 ধাপ ২/৩: ছবি ও কবিতার অডিও লোড হচ্ছে...")
print("====================================================================")
img_url = "https://raw.githubusercontent.com/kaalikolom-ux/video-project/main/assets/poem_portrait.jpg?v=elevenlabs"
audio_url = "https://raw.githubusercontent.com/kaalikolom-ux/video-project/main/assets/poem_recitation.mp3?v=elevenlabs"
urllib.request.urlretrieve(img_url, "/content/source_image.jpg")
urllib.request.urlretrieve(audio_url, "/content/driven_audio.mp3")
!ffmpeg -y -i /content/driven_audio.mp3 -ar 16000 /content/driven_audio.wav 2>/dev/null
print("✅ আপনার ছবি (source_image.jpg) এবং কবিতার অডিও (driven_audio.wav) লোড সম্পন্ন!")

print("\n====================================================================")
print("🎬 ধাপ ৩/৩: এআই লিপ-সিঙ্ক ভিডিও তৈরি শুরু হচ্ছে... (অনুগ্রহ করে অপেক্ষা করুন)")
print("====================================================================")
!rm -rf /content/results
!mkdir -p /content/results

# Run inference with full portrait and face enhancer (GFPGAN)
!python inference.py \
    --driven_audio /content/driven_audio.wav \
    --source_image /content/source_image.jpg \
    --result_dir /content/results \
    --still \
    --preprocess full \
    --enhancer gfpgan

# Find generated video (recursive check)
video_files = glob.glob("/content/results/**/*.mp4", recursive=True)
if not video_files:
    video_files = glob.glob("/content/results/*.mp4")

if video_files:
    enhanced_files = [f for f in video_files if "_enhanced" in f]
    latest_video = max(enhanced_files, key=os.path.getctime) if enhanced_files else max(video_files, key=os.path.getctime)
    
    print(f"\n🎉 অভিনন্দন! আপনার ভিডিও সফলভাবে তৈরি হয়েছে: {latest_video}")
    
    # Preview video in notebook
    mp4 = open(latest_video, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'''
    <div style="text-align: center; margin: 20px 0;">
      <video width=540 controls autoplay style="border-radius:14px; box-shadow: 0 10px 30px rgba(0,0,0,0.6);">
          <source src="{data_url}" type="video/mp4">
      </video>
    </div>
    '''))
    
    # Trigger auto-download to PC
    print("⬇️ কম্পিউটারে ভিডিও ডাউনলোড শুরু হচ্ছে...")
    files.download(latest_video)
else:
    print("❌ কোনো ভিডিও খুঁজে পাওয়া যায়নি। দয়া করে এরর লগ চেক করুন।")